[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agustrionopj/python-mini-workshop/blob/main/scientific_images_as_data.ipynb?forcePlay=true)

# Pengolahan citra dengan `Python`

**Mini-workshop:** _Pelatihan Python — Imahnoong_  
**Pemateri:** Agus T. P. Jatmiko (Observatorium Bosscha)  
**Durasi:** ± 3 jam · **Tanggal:** 2026-10-31

Notebook ini dirancang untuk peserta **tanpa latar belakang astronomi**.
Lima modul ringkas (± 140 menit + jeda):

| Modul | Topik | Durasi |  |
|---|---|---|---|
| 1 | Citra ilmiah sebagai data | 25′ | NumPy · Astropy · Pandas |
| 2 | Inspeksi & karakterisasi statistik | 25′ | NumPy · Astropy · Matplotlib |
| 3 | Kalibrasi & koreksi | 35′ | ccdproc · Astropy · NumPy |
| 4 | Reduksi derau & citra berwarna | 25′ | NumPy · Astropy |
| 5 | Pengukuran & alur kerja reproduktibel | 30′ | Pandas · Astropy · NumPy |
| 6 | Overscan | 30′ | Pandas · Astropy · NumPy |

Setiap modul ditutup dengan **✋ Hands-on**, kerjakan di versi salinan anda sendiri.

## ⚠️ Jangan edit file ini secara langsung!

Anda membuka notebook ini dari **file asli** pada GitHub. Gunakan dalam mode
_read-only_:

- Anda bisa menjalankan seluruh _cell_ secara bebas dan mandiri.
- Jika ingin melakukan perubahan: `File → Save a copy in Drive` (simpan ke
  Drive anda sendiri).
- File asli pada repository tidak akan melihat perubahan yang anda lakukan.

Semua proses berlangsung di dalam Colab, tidak perlu memasang apa pun.

In [ ]:
# --- Setup ---
# !pip install -q astropy ccdproc pandas numpy matplotlib pillow

import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from astropy.io import fits
import astropy.units as u
import ccdproc

plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 100
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
print('Astropy:', __import__('astropy').__version__)

In [ ]:
# --- Ambil data dari repository workshop ---
DATA_BASE_URL = "https://raw.githubusercontent.com/agustrionopj/python-mini-workshop/main/data"

# data khusus notebook ini
BASIC_FILES = [
    "mock_basics/color_photo.png",
    "mock_basics/testpattern_001.fit",
    "mock_basics/problem_001.fit",
    "mock_basics/clean_001.fit",
    "mock_basics/field-r-001.fit",
    "mock_basics/field-g-001.fit",
    "mock_basics/field-b-001.fit",
]

# data CCD (dipakai bersama notebook ccd_preprocessing_astropy)
BIAS_FILES    = [f"mock_nooverscan/bias/bias-mock-{i:03d}.fit" for i in range(1, 11)]
DARK_FILES    = ([f"mock_nooverscan/dark/dark-mock-0002s-{i:03d}.fit" for i in range(1, 6)]
               + [f"mock_nooverscan/dark/dark-mock-0120s-{i:03d}.fit" for i in range(1, 6)])
FLAT_FILES    = [f"mock_nooverscan/flat/flat-mock-v-2s-{i:03d}.fit" for i in range(1, 11)]
SCIENCE_FILES = [f"mock_nooverscan/science/field-mock-v-120s-{i:03d}.fit" for i in range(1, 6)]

# data overscan (Modul 6)
OVERSCAN_FILES = ([f"mock_basics/overscan/bias/bias-mock-ov-{i:03d}.fit" for i in range(1, 6)]
                + [f"mock_basics/overscan/dark/dark-mock-ov-0120s-{i:03d}.fit" for i in range(1, 6)]
                + [f"mock_basics/overscan/flat/flat-mock-ov-v-2s-{i:03d}.fit" for i in range(1, 6)]
                + [f"mock_basics/overscan/science/field-mock-ov-v-120s-{i:03d}.fit" for i in range(1, 3)])

ALL_FILES = (BASIC_FILES + BIAS_FILES + DARK_FILES + FLAT_FILES + SCIENCE_FILES + OVERSCAN_FILES)
print(f"Total file: {len(ALL_FILES)}")

In [ ]:
# --- Unduh ---
for relpath in ALL_FILES:
    target = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        !wget -q -O {target} {DATA_BASE_URL}/{relpath}
        print("Downloaded:", relpath)

print("\nBerkas tersedia:")
!find {DATA_DIR} -name "*.fit*" | sort

## Modul 1 - Citra sebagai data ilmiah (25 menit)

**Poin yang perlu dipahami:**

1. Semua citra digital (foto ponsel, hasil mikroskop, detektor teleskop, dsb)
   pada akhirnya hanyalah **array angka** (`numpy`).
2. Setiap angka = **nilai satu piksel**; makna fisiknya dijelaskan oleh
   **metadata** (header).
3. Citra **monokrom** = array 2D `(baris, kolom)`. Citra **berwarna/RGB** =
   array 3D `(baris, kolom, kanal)`.
4. **FITS** adalah format standar sains (astronomi, bio-imaging, dsb.) karena
   menyimpan **data + metadata + unit** dalam satu file.

In [ ]:
# --- Array NumPy: "citra" paling sederhana ---
import numpy as np

arr = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]], dtype=np.float32)

In [ ]:
print("array:\n", arr)
print("shape  :", arr.shape)        # (baris, kolom)
print("dtype  :", arr.dtype)        # tipe data setiap angka
print("nilai [0,1] (baris 0, kolom 1):", arr[0, 1])

# "piksel" = satu elemen array
arr[1, 1] = 99                      # ubah satu piksel
print("setelah diubah:\n", arr)

# rentang tipe data itu penting: satu piksel bisa overflow kalau tipe salah!
for dt in (np.uint8, np.uint16, np.float32):
    info = np.iinfo(dt) if dt in (np.uint8, np.uint16) else np.finfo(dt)
    print(f"{np.dtype(dt).name:8s} min={info.min}  max={info.max}")

In [ ]:
# --- Membaca citra FITS dengan Astropy ---
import os
from astropy.io import fits

DATA_DIR = 'data'
SAMPLE = os.path.join(DATA_DIR, 'mock_basics/clean_001.fit')

with fits.open(SAMPLE) as hdul:
    hdul.info()                        # struktur file (HDU)
    header = hdul[0].header.copy()     # metadata
    sci = hdul[0].data                 # data piksel → array numpy

print("\nshape:", sci.shape, "| dtype:", sci.dtype)
header   # tampil sebagai tabel HTML rapi di Colab

### Header, metadata, dan unit

Header FITS = daftar `KEYWORD = value`. Yang umum dijumpai:

| Keyword | Arti | Contoh di data kita |
|---|---|---|
| `IMAGETYP` | jenis frame | `SCIENCE`, `BIAS`, `DARK`, `FLAT` |
| `EXPTIME` | lama eksposur (detik) | `120.0` |
| `FILTER` | filter optik | `V` (visual) |
| `GAIN` | konversi e⁻ → ADU | `1.0` |
| `BUNIT` | **unit** nilai piksel | `adu` |
| `DATE-OBS` | waktu pengamatan | `2026-10-30T…` |

**Penting:** array angka tanpa header tidak bermakna ilmiah, kita tidak tahu
nilainya bersatuan apa, kapan dan bagaimana diambil.

In [ ]:
# --- Citra biasa (PNG/JPEG) dengan Pillow ---
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100

PNG_PATH = os.path.join(DATA_DIR, 'mock_basics/color_photo.png')
photo = Image.open(PNG_PATH)
print("objek PIL :", photo.format, photo.size, photo.mode)

photo_arr = np.array(photo)            # konversi ke array numpy
print("array numpy :", photo_arr.shape, "| dtype:", photo_arr.dtype)
# shape (480, 640, 3) = (baris, kolom, kanal RGB); tiap kanal 0–255

plt.figure(figsize=(6, 4))
plt.imshow(photo_arr)
plt.title('Citra RGB (Pillow → NumPy)')
plt.axis('off')
plt.show()

In [ ]:
# --- Monokrom vs RGB, berdampingan ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(sci, cmap='gray', origin='lower')
axes[0].set_title('FITS monokrom (2D)')
fig.colorbar(im0, ax=axes[0], label=header.get('BUNIT', 'ADU'))

axes[1].imshow(photo_arr)
axes[1].set_title('PNG RGB (3 kanal)')
axes[1].axis('off')
plt.show()

print("Keduanya array numpy! Bedanya hanya dimensi:",
      sci.shape, "vs", photo_arr.shape)

### Pandas untuk metadata eksposur

Kalau file FITS yang dimiliki ada beberapa, jangan buka satu per satu secara manual.
`pandas.DataFrame` mengubah kumpulan header menjadi **tabel** yang bisa
diurutkan, difilter, dan diringkas, memberikan pola dasar alur kerja ilmiah yang bisa direproduksi.

In [ ]:
# --- Kumpulkan header menjadi tabel Pandas ---
import glob
from IPython.display import display

CCD_PATHS = sorted(glob.glob(os.path.join(DATA_DIR, 'mock_nooverscan/*/*.fit')))
rows = []
for path in CCD_PATHS:
    hdr = fits.getheader(path)
    rows.append({
        'file': os.path.basename(path),
        'jenis': hdr['IMAGETYP'],
        'exptime_s': hdr['EXPTIME'],
        'filter': hdr['FILTER'],
        'temp_C': hdr['SET-TEMP'],
        'date': hdr['DATE-OBS'][:19],
    })

meta = pd.DataFrame(rows)
print(f"{len(meta)} frame ditemukan")
meta.head(8)

In [ ]:
# --- Ringkas tabel ---
display(meta.describe())                                       # kolom numerik
display(meta.groupby('jenis')['exptime_s'].agg(['count', 'min', 'max']))

### ✋ Hands-on 1 (± 5 menit)

Dengan pola yang sama seperti contoh di atas:

1. Tampilkan `mock_basics/problem_001.fit`,  periksa `shape` dan `dtype`-nya.
   Kenapa tipe datanya beda dengan `clean_001.fit`?
2. Pilih 2 keyword header yang menurut anda paling penting, jelaskan singkat.
3. Tampilkan `problem_001.fit` (abu-abu) **bersebelahan** dengan
   `color_photo.png`.

In [ ]:
# TODO 1: muat problem_001.fit, periksa shape & dtype
# prob = fits.getdata(...)
# print(prob.shape, prob.dtype)

# TODO 2: dua keyword header favorit anda

# TODO 3: tampil bersebelahan (fig, axes = plt.subplots(1, 2, ...))

## Modul 2 — Inspeksi & karakterisasi statistik (25 menit)

Sebelum diolah, ketahui dulu **karakter** citranya:

- statistik dasar: `min`, `max`, `mean`, `median`, `std`;
- **histogram** = distribusi intensitas (langit? bintang? derau?);
- nilai **NaN** (piksel mati), **jenuh / saturated** (mentok di 65535 untuk
  data 16-bit), dan **pencilan** harus ditangani dengan baik;
- **mask** = "peta" piksel yang kita abaikan dalam perhitungan;
- **tampilan ≠ data**: peregangan kontras hanya mengubah tampilan — nilai
  ilmiah di dalam array tidak boleh diubah.

In [ ]:
# --- Muat frame "bermasalah" dan hitung statistik ---
PROB = os.path.join(DATA_DIR, 'mock_basics/problem_001.fit')
prob = fits.getdata(PROB).astype(float)

n_nan = int(np.isnan(prob).sum())
n_sat = int((prob >= 65535).sum())

print(f"shape={prob.shape} | NaN={n_nan} piksel | saturated={n_sat} piksel\n")

# HATI-HATI: statistik biasa "rusak" kalau ada NaN
print("np.mean     :", np.mean(prob))        # → nan!
print("np.nanmean  :", np.nanmean(prob))
print("np.median   :", np.median(prob))       # → nan!
print("np.nanmedian:", np.nanmedian(prob))
print("np.nanstd   :", np.nanstd(prob))

In [ ]:
# --- Histogram distribusi intensitas ---
valid = prob[~np.isnan(prob)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(valid.ravel(), bins=200, range=(100, 3000))
axes[0].set_yscale('log')
axes[0].set_xlabel('ADU')
axes[0].set_title('problem_001: mayoritas = langit + derau')

axes[1].hist(sci.ravel(), bins=100, range=(0, 3000))   # sci = clean_001.fit (dimuat di Modul 1)
axes[1].set_xlabel('ADU')
axes[1].set_title('clean_001 (latar belakang berbeda!)')
plt.show()

In [ ]:
# --- Mask: NaN + saturated ---
mask_bad = np.isnan(prob) | (prob >= 65535)
print("piksel buruk:", int(mask_bad.sum()),
      f"({100 * mask_bad.mean():.3f}% dari frame)")

# np.ma: array bermask,  statistik otomatis mengabaikan piksel buruk
prob_ma = np.ma.masked_array(prob, mask=mask_bad)
print("np.ma.mean:", float(prob_ma.mean()), "| np.ma.std:", float(prob_ma.std()))

# visual: piksel buruk ditandai merah
fig, ax = plt.subplots(figsize=(7, 7))
lo, hi = np.nanpercentile(prob, (5, 95))
ax.imshow(prob, cmap='gray', origin='lower', vmin=lo, vmax=hi)
overlay = np.zeros((*prob.shape, 4))
overlay[mask_bad] = (1, 0, 0, 1)
ax.imshow(overlay, origin='lower')
ax.set_title('merah = NaN / saturated')
plt.show()

In [ ]:
# --- Tampilan vs nilai ilmiah ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(prob, cmap='gray', vmin=0, vmax=65535, origin='lower')
axes[0].set_title('linear penuh 0–65535 → "hitam"')

lo, hi = np.nanpercentile(prob, (5, 95))
axes[1].imshow(prob, cmap='gray', vmin=lo, vmax=hi, origin='lower')
axes[1].set_title('persentil 5–95% → "terlihat"')

axes[2].imshow(np.log10(np.where(prob > 0, prob, np.nan)),
               cmap='gray', origin='lower')
axes[2].set_title('skala log → struktur redup muncul')
plt.show()

# tetapi nilai datanya TIDAK berubah oleh tampilan:
print("median data tetap:", np.nanmedian(prob))

In [ ]:
# --- Bandingkan dua citra secara statistik ---
stats = pd.DataFrame({
    'clean_001': [np.min(sci), np.median(sci), np.mean(sci),
                  sci.std(), sci.max()],
    'problem_001': [np.nanmin(prob), np.nanmedian(prob), np.nanmean(prob),
                    np.nanstd(prob), np.nanmax(prob)],
}, index=['min', 'median', 'mean', 'std', 'max'])
display(stats)

# citra selisih: di mana perbedaannya paling besar?
diff = sci - np.nan_to_num(prob, nan=0.0)   # area NaN tampil "merah"
diff_std = diff.std()

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(diff, cmap='coolwarm', origin='lower',
               vmin=-5 * diff_std, vmax=+5 * diff_std)
fig.colorbar(im, label='selisih (ADU)')
ax.set_title('clean − problem')
plt.show()

### ✋ Hands-on 2 (± 5 menit)

1. Berapa persen piksel `problem_001` yang **jenuh/saturasi**, dan di koordinat mana?
   (petunjuk: `np.argwhere`)
2. Di baris/kolom berapa "kotak" NaN berada? (cocokkan dengan tanda merah di
   atas)
3. Gambarkan histogram kedua citra dalam **satu sumbu** (`plt.hist` dua kali). Apa arti perbedaan median kedua frame?

In [ ]:
# TODO 1: koordinat piksel jenuh
# sat_pos = np.argwhere(prob >= 65535)
# print(sat_pos[:10])

# TODO 2: batas balok NaN (argwhere → nilai min/max tiap dimensi)

# TODO 3: dua histogram dalam satu sumbu

## Modul 3 — Kalibrasi & koreksi (35 menit)

Citra ilmiah selalu "tercemar" oleh perilaku instrumennya. Analogi
sehari-hari: foto ponsel pun otomatis menjalankan koreksi lensa & derau. 
Dalam sains kita melakukannya **secara eksplisit dan terukur**.

Tiga koreksi standar untuk detektor CCD (berlaku juga untuk sensor kamera,
kamera mikroskop, detektor sinar-X, dll.):

| Frame | Mengukur apa | Koreksi |
|---|---|---|
| **BIAS** | level nol elektronika (eksposur 0 detik) | dikurangkan |
| **DARK** | sinyal termal (sebanding dengan waktu eksposur) | dikurangkan, diskalakan ke eksposur |
| **FLAT** | variasi sensitivitas piksel + efek optik | dibagikan (setelah dinormalisasi) |


$$
\boxed{
\mathrm{Bersih} = \frac{(\text{Science - Bias - Dark * t\_\text{exp}})}{\text{Flat}}
}
$$

Frame kalibrasi **digabung** (median) dari banyak frame agar derau turun
$= \sqrt{N}$ dan kejadian acak (misalnya sinar kosmik) menjadi hilang. Paket `ccdproc`
membungkus seluruh proses ini.

In [ ]:
# --- Susun master bias / dark / flat dengan ccdproc.Combiner ---
import astropy.units as u
import ccdproc


def load_ccd(relpath):
    return ccdproc.CCDData.read(os.path.join(DATA_DIR, relpath), unit='adu')


bias_ccds = [load_ccd(f) for f in BIAS_FILES]
dark_ccds = [load_ccd(f) for f in DARK_FILES if '0120s' in f]   # kelompok 120 s
flat_ccds = [load_ccd(f) for f in FLAT_FILES]


def median_combine(ccds):
    comb = ccdproc.Combiner(ccds)
    # (opsional) tolak pencilan sebelum penggabungan:
    # comb.sigma_clipping(low_thresh=5, high_thresh=5)
    return comb.median_combine()


master_bias = median_combine(bias_ccds)

master_dark = median_combine(dark_ccds).subtract(master_bias)   # dark murni (tanpa bias)
master_dark.meta['EXPTIME'] = 120.0        # eksposur kelompok dark yang dipakai

master_flat_raw = median_combine(flat_ccds).subtract(master_bias)
master_flat = ccdproc.CCDData(
    master_flat_raw.data / np.median(master_flat_raw.data),
    unit=u.dimensionless_unscaled)         # dinormalisasi: median = 1

print("master bias :", master_bias.shape, "median", float(np.median(master_bias.data)))
print("master dark :", master_dark.shape, "median", float(np.median(master_dark.data)))
print("master flat :", master_flat.shape, "median", float(np.median(master_flat.data)))

In [ ]:
# --- Kalibrasi satu frame science dengan ccdproc ---
ccd_sci = load_ccd(SCIENCE_FILES[0])

# 1) kurangi bias
ccd_cal = ccdproc.subtract_bias(ccd_sci, master_bias)

# 2) kurangi dark, diskalakan ke eksposur science (header EXPTIME dibaca otomatis)
ccd_cal = ccdproc.subtract_dark(ccd_cal, master_dark,
                                exposure_time='EXPTIME',
                                exposure_unit=u.second,
                                scale=True)

# 3) koreksi flat
ccd_cal = ccdproc.flat_correct(ccd_cal, master_flat)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(ccd_sci.data, cmap='gray', origin='lower',
               vmin=np.percentile(ccd_sci.data, 5),
               vmax=np.percentile(ccd_sci.data, 95))
axes[0].set_title('RAW')
axes[1].imshow(ccd_cal.data, cmap='gray', origin='lower',
               vmin=np.percentile(ccd_cal.data, 5),
               vmax=np.percentile(ccd_cal.data, 95))
axes[1].set_title('TERKALIBRASI (ccdproc)')
plt.show()

print("median latar belakang: raw =", float(np.median(ccd_sci.data)),
      "→ kalibrasi =", float(np.median(ccd_cal.data)))

### Ketidakpastian (uncertainty) & mask dalam kalibrasi

`ccdproc` dapat membawa **estimasi ketidakpastian per piksel** (dari derau
pembacaan + derau Poisson), dan operasi aritmatika mempropagasikannya secara
konsisten.

**Mask** juga turut terbawa dalam perhitungan: piksel bertanda `mask=True` diabaikan dalam operasi
berikutnya.

### Lingkup aplikasi: Astronomi, non-Astronomi

Matematika di balik proses kalibrasi di atas identik untuk:
- sensor ponsel / DSLR (koreksi lensa shading & derau),
- kamera mikroskop dan endoskop (flat = iluminasi lapangan),
- detektor sinar-X, inframerah termal, kamera kendaraan otonom.

Yang berbeda hanya cara memperoleh frame kalibrasinya.

In [ ]:
# --- Ketidakpastian + mask ---
# konstanta kamera (konsisten dengan header GAIN/RDNOISE pada data mock):
GAIN = 1.0        # e-/ADU
READ_NOISE = 5.0  # e-

dev = ccdproc.create_deviation(ccd_sci,
                               gain=GAIN * u.electron / u.adu,
                               readnoise=READ_NOISE * u.electron)
print("ketidakpastian per piksel (5 contoh):", dev.uncertainty.array.ravel()[:5])
print("rata-rata ketidakpastian:", float(dev.uncertainty.array.mean()), "ADU")

# mask: buang piksel dengan sensitivitas flat paling rendah (persentil terbawah)
batas_flat = np.percentile(master_flat.data, 0.5)
ccd_cal.mask = master_flat.data < batas_flat
print(f"piksel dengan mask (flat < {batas_flat:.3f}):", int(ccd_cal.mask.sum()))

# operasi berikutnya otomatis mematuhi mask:
hasil = ccd_cal.add(master_bias)
print("mask ikut terbawa:", hasil.mask is not None and bool(hasil.mask.sum()))

### ✋ Hands-on 3 (± 8 menit)

Kalibrasi **frame science kedua** dengan alur yang sama:

1. `subtract_bias` → `subtract_dark` (scale) → `flat_correct`.
2. Catat median latar belakang sebelum vs sesudah.
3. (Bonus) tampilkan raw vs terkalibrasi bersebelahan.

In [ ]:
# TODO: kalibrasi frame science kedua
# ccd_sci2 = load_ccd(SCIENCE_FILES[1])
# ...

## Modul 4 — Reduksi derau & penanganan citra berwarna (25 menit)

Tiga istilah yang wajib dibedakan:

- **Sinyal** $→$ informasi yang dicari (bintang, struktur jaringan, objek);
- **latar (background)** $→$ level dasar (langit, iluminasi ruangan);
- **derau (noise)** $→$ fluktuasi acak di sekitar nilai sebenarnya.

_Tools_ sederhana yang ampuh:

- **median filter** $→$ menghapus pencilan / hot-pixel tanpa mengaburkan tepi
  seburuk filter rata-rata (mean);
- **masking pencilan** $\rightarrow$ nilai ekstrem diganti median sekitarnya.

Citra berwarna diperlakukan sebagai **3 kanal terpisah**: statistik per
kanal, normalisasi per kanal, lalu digabung kembali.

**Penting:** _stretch_ hanya untuk *tampilan*, TIDAK boleh
menggantikan kalibrasi ilmiah (Modul 3).

In [ ]:
# --- Median filter & masking pencilan pada citra uji ---
# "citra uji" (test pattern) tanpa bintang: pola halus + hot-pixel buatan
TP = os.path.join(DATA_DIR, 'mock_basics/testpattern_001.fit')
tp = fits.getdata(TP).astype(float)

tp_med = ccdproc.median_filter(ccdproc.CCDData(tp, unit='adu'), size=3).data

# deteksi pencilan: selisih piksel dengan median lingkungannya (residual)
residual = tp - tp_med
resid_std = 1.4826 * np.median(np.abs(residual - np.median(residual)))
outlier = residual > 10 * resid_std            # hot-pixel = pencilan positif
tp_fixed = np.where(outlier, tp_med, tp)
print(f"resid_std = {resid_std:.1f} | piksel pencilan dibersihkan: {int(outlier.sum())}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(tp, cmap='gray', origin='lower', vmin=100, vmax=2500)
axes[0].set_title('sebelum: hot-pixel + pencilan')
axes[1].imshow(tp_fixed, cmap='gray', origin='lower', vmin=100, vmax=2500)
axes[1].set_title('sesudah median filter 3×3 + masking')
plt.show()

# TODO (diskusi): piksel di tepi kotak/lingkaran juga bisa terflag —
# kapan masking pencilan aman, kapan berbahaya?

In [ ]:
# --- RGB sebagai 3 kanal terpisah ---
r, g, b = photo_arr[..., 0], photo_arr[..., 1], photo_arr[..., 2]

ch_stats = pd.DataFrame({
    'R': [r.min(), r.mean(), np.median(r), r.std(), r.max()],
    'G': [g.min(), g.mean(), np.median(g), g.std(), g.max()],
    'B': [b.min(), b.mean(), np.median(b), b.std(), b.max()],
}, index=['min', 'mean', 'median', 'std', 'max'])
display(ch_stats)


def normalize(ch):
    """Normalisasi min–max per kanal (hanya untuk tampilan!)."""
    ch = ch.astype(float)
    return (ch - ch.min()) / (ch.max() - ch.min())


fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (nama, ch) in zip(axes, [('R', r), ('G', g), ('B', b)]):
    ax.imshow(normalize(ch), cmap='gray')
    ax.set_title(f'kanal {nama}')
    ax.axis('off')
plt.show()

In [ ]:
# --- Bonus: komposit warna dari 3 frame monokrom FITS ---
from astropy.visualization import make_lupton_rgb

r_ccd = fits.getdata(os.path.join(DATA_DIR, 'mock_basics/field-r-001.fit')).astype(float)
g_ccd = fits.getdata(os.path.join(DATA_DIR, 'mock_basics/field-g-001.fit')).astype(float)
b_ccd = fits.getdata(os.path.join(DATA_DIR, 'mock_basics/field-b-001.fit')).astype(float)

# kurangi latar belakang tiap kanal dulu (mini-kalibrasi!)
r_ccd -= np.median(r_ccd)
g_ccd -= np.median(g_ccd)
b_ccd -= np.median(b_ccd)
r_ccd = np.clip(r_ccd, 0, None)      # nilai negatif tak bermakna untuk tampilan
g_ccd = np.clip(g_ccd, 0, None)
b_ccd = np.clip(b_ccd, 0, None)

rgb = make_lupton_rgb(r_ccd, g_ccd, b_ccd, Q=10, stretch=2.0)

plt.figure(figsize=(6, 6))
plt.imshow(rgb, origin='lower')
plt.title('Komposit warna: bintang "merah" vs "biru"')
plt.axis('off')
plt.show()
# TODO (diskusi): kenapa sebagian bintang tampak merah dan sebagian biru?

### ✋ Hands-on 4 (± 5 menit)

1. Ubah ambang deteksi pencilan dari `10 × resid_std` menjadi `5 ×` dan
   `20 ×` — bagaimana jumlah piksel yang dibersihkan? Kapan masking aman,
   kapan berbahaya (piksel tepi objek ikut "tersapu")?
2. Untuk `color_photo.png`: kanal mana kontrasnya paling besar?
   (bandingkan `std` ketiga kanal)

In [ ]:
# TODO 1: potongan 40×40 sebelum/sesudah median filter
# zoom = (slice(...), slice(...))

# TODO 2: std per kanal
# print(r.std(), g.std(), b.std())

## Modul 5 — Pengukuran & alur kerja (30 menit)

Ilmiah berarti **terukur dan dapat diulang**. Modul ini merangkai semuanya:

1. ukur (statistik ROI)
2. simpan hasil dalam `DataFrame`
3. bandingkan raw vs terkalibrasi
4. ekspor (FITS + CSV)
5. bungkus menjadi **fungsi alur kerja** yang dapat dipakai ulang.

In [ ]:
# --- Pengukuran region-of-interest (ROI) ---
cal = ccd_cal.data                        # hasil kalibrasi Modul 3

# (a) ROI kotak: cukup potongan array
roi_box = cal[300:400, 100:200]

# (b) ROI lingkaran di sekitar piksel terterang (bintang)
yy, xx = np.mgrid[0:cal.shape[0], 0:cal.shape[1]]
y_star, x_star = np.unravel_index(np.argmax(cal), cal.shape)
radius = 8.0
roi_star = cal[(xx - x_star) ** 2 + (yy - y_star) ** 2 < radius ** 2]

print(f"bintang di (x={x_star}, y={y_star})")
print(f"ROI kotak   : median={np.median(roi_box):.1f}  std={roi_box.std():.1f}")
print(f"ROI bintang : flux={roi_star.sum():.0f} ADU  "
      f"puncak={roi_star.max():.1f}  latar ~{np.median(cal):.1f}")

plt.imshow(cal, cmap='gray')
plt.plot(x_star, y_star, 'ro')
plt.show()

In [ ]:
# --- Simpan pengukuran dalam DataFrame ---
rows = []
for f in SCIENCE_FILES:
    path = os.path.join(DATA_DIR, f)
    d = fits.getdata(path).astype(float)
    rows.append({
        'file': os.path.basename(f),
        'bg_median_raw': float(np.median(d)),
        'bg_std_raw': float(d.std()),
        'max_raw': float(d.max()),
        'exptime_s': fits.getheader(path)['EXPTIME'],
    })
meas = pd.DataFrame(rows)
meas['bg_median_cal'] = np.nan       # diisi oleh alur kerja di bawah
display(meas)

In [ ]:
# --- Ekspor: FITS terkalibrasi + tabel CSV ---
out_hdr = ccd_sci.header.copy()
out_hdr['HISTORY'] = 'kalibrasi: master bias + dark (skala eksposur) + flat'
out_hdr['HISTORY'] = 'scientific_images_as_data.ipynb (mini-workshop)'

fits.PrimaryHDU(data=cal, header=out_hdr).writeto(
    os.path.join(DATA_DIR, 'hasil_kalibrasi_001.fits'), overwrite=True)

meas.to_csv('hasil_pengukuran.csv', index=False)
print("tersimpan: data/hasil_kalibrasi_001.fits dan hasil_pengukuran.csv")

# unduh dari Colab: panel Files (kiri), atau:
# from google.colab import files; files.download('hasil_pengukuran.csv')

In [ ]:
# --- Alur kerja reproduktibel: satu fungsi untuk semuanya ---
def preprocess(path, master_bias, master_dark, master_flat):
    """Load → kalibrasi (bias, dark, flat) → ukur.

    Kembalikan (CCDData terkalibrasi, dict ringkasan).
    """
    ccd = ccdproc.CCDData.read(path, unit='adu')
    ccd = ccdproc.subtract_bias(ccd, master_bias)
    ccd = ccdproc.subtract_dark(ccd, master_dark, exposure_time='EXPTIME',
                                exposure_unit=u.second, scale=True)
    ccd = ccdproc.flat_correct(ccd, master_flat)

    d = ccd.data
    summary = {
        'file': os.path.basename(path),
        'bg_median_cal': float(np.median(d)),
        'bg_std_cal': float(d.std()),
        'max_cal': float(d.max()),
    }
    return ccd, summary


results = []
for f in SCIENCE_FILES:
    _, s = preprocess(os.path.join(DATA_DIR, f),
                      master_bias, master_dark, master_flat)
    results.append(s)

results_df = pd.DataFrame(results)
display(results_df)

### ✋ Hands-on 5 (± 5 menit)

1. Tambahkan kolom `'snr'` ke `results_df` — signal-to-noise sederhana:
   `bg_median / bg_std`, atau gunakan flux bintang dari sel ROI.
2. Simpan `results_df` versi anda sebagai `hasil_kelompok_<nama>.csv`.
3. (Bonus) ubah `preprocess()` agar ringkasan juga memuat `n_masked`
   (banyak piksel yang bermask).

In [ ]:
# TODO 1: kolom SNR
# results_df['snr'] = ...

# TODO 2: ekspor CSV versi kelompok anda
# results_df.to_csv(...)

# TODO 3: perluas preprocess()

## Modul 6 — Overscan: koreksi level bias per baris (± 30 menit)

Banyak detektor CCD membaca beberapa **kolom ekstra** setelah area citra:
**strip overscan**. Kolom ini tidak melihat cahaya — ia hanya merekam
**level bias** elektronik saat frame itu dibaca.

Kenapa penting? Level bias bisa **bergeser antar frame** (suhu elektronik,
waktu pembacaan), bahkan **sepanjang baris** di dalam satu frame. Master bias
dari frame lain tidak menangkap geseran frame ini — strip overscan dari
frame itu sendiri bisa!

| Keyword header | Arti | Contoh |
|---|---|---|
| `DATASEC` | area citra | `[1:512,1:512]` |
| `BIASSEC` | strip overscan (16 kolom) | `[513:528,1:512]` |
| `TRIMSEC` | area yang disimpan setelah trim | `= DATASEC` |

Strategi standar:

1. untuk **tiap baris**, ukur median strip overscan baris itu;
2. kurangkan dari seluruh baris (koreksi per baris);
3. **trim**: buang kolom overscan — citra kembali 512×512.

Data set: `data/mock_basics/overscan/` — 5 bias, 5 dark (120 s),
5 flat (2 s V), 2 science (120 s V).

In [ ]:
# --- Inspeksi frame overscan ---
OV_DIR = os.path.join(DATA_DIR, 'mock_basics/overscan')

bias_ov = fits.getdata(os.path.join(OV_DIR, 'bias/bias-mock-ov-001.fit')).astype(float)
print("shape:", bias_ov.shape, "- 512 baris x (512 citra + 16 overscan) kolom")

image = bias_ov[:, :512]        # area citra
strip = bias_ov[:, 512:]        # strip overscan
print(f"median strip     : {np.median(strip):.1f}  (level bias elektronik)")
print(f"median area citra: {np.median(image):.1f}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(bias_ov, cmap='gray', origin='lower', vmin=490, vmax=515)
ax.axvline(511.5, color='r', lw=2.5)
ax.set_title('merah = batas strip overscan (16 kolom di kanan)')
plt.show()

In [ ]:
# --- Cara manual (numpy): koreksi per baris + trim ---
# 1) level bias tiap baris = median strip baris itu
row_level = np.median(strip, axis=1)        # 512 nilai, satu per baris

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(row_level)
ax.set_xlabel('baris')
ax.set_ylabel('median overscan (ADU)')
ax.set_title('profil level bias sepanjang baris')
plt.show()

# geseran antar frame terlihat langsung:
for i in range(1, 6):
    d = fits.getdata(os.path.join(OV_DIR, f'bias/bias-mock-ov-{i:03d}.fit')).astype(float)
    print(f"  frame ov-{i:03d}: median strip = {np.median(d[:, 512:]):.1f}")

# 2) kurangi tiap baris dengan level baris itu, 3) buang kolom overscan
bias_manual = image - row_level[:, None]
print("shape:", bias_ov.shape, "-> trim ->", bias_manual.shape)

In [ ]:
# --- Cara ccdproc: subtract_overscan + trim_image ---
ccd_ov = ccdproc.CCDData.read(os.path.join(OV_DIR, 'bias/bias-mock-ov-001.fit'),
                              unit='adu')
print("header: DATASEC =", ccd_ov.header['DATASEC'],
      "| BIASSEC =", ccd_ov.header['BIASSEC'])

# median=True + overscan_axis=1: satu nilai median per baris, seperti cara manual
ccd_ovc = ccdproc.subtract_overscan(ccd_ov, fits_section=ccd_ov.header['BIASSEC'],
                                    median=True, overscan_axis=1)
ccd_ovc = ccdproc.trim_image(ccd_ovc, fits_section=ccd_ov.header['TRIMSEC'])

bias_ccdproc = ccd_ovc.data
print("shape setelah trim:", bias_ccdproc.shape)

# hasil harus identik dengan cara manual:
print("selisih maksimum manual vs ccdproc:",
      float(np.abs(bias_manual - bias_ccdproc).max()))

In [ ]:
# --- Mini-pipeline: overscan -> master -> kalibrasi ---
def load_ov(relpath):
    """Muat frame overscan: koreksi strip per baris + trim."""
    ccd = ccdproc.CCDData.read(os.path.join(OV_DIR, relpath), unit='adu')
    ccd = ccdproc.subtract_overscan(ccd, fits_section=ccd.header['BIASSEC'],
                                    median=True, overscan_axis=1)
    return ccdproc.trim_image(ccd, fits_section=ccd.header['TRIMSEC'])


bias_ovs = [load_ov(f'bias/bias-mock-ov-{i:03d}.fit') for i in range(1, 6)]
dark_ovs = [load_ov(f'dark/dark-mock-ov-0120s-{i:03d}.fit') for i in range(1, 6)]
flat_ovs = [load_ov(f'flat/flat-mock-ov-v-2s-{i:03d}.fit') for i in range(1, 6)]
sci_ovs  = [load_ov(f'science/field-mock-ov-v-120s-{i:03d}.fit') for i in range(1, 3)]

master_bias_ov = ccdproc.Combiner(bias_ovs).median_combine()
master_dark_ov = ccdproc.Combiner(dark_ovs).median_combine().subtract(master_bias_ov)
master_dark_ov.meta['EXPTIME'] = 120.0
flat_raw_ov = ccdproc.Combiner(flat_ovs).median_combine().subtract(master_bias_ov)
master_flat_ov = ccdproc.CCDData(flat_raw_ov.data / np.median(flat_raw_ov.data),
                                 unit=u.dimensionless_unscaled)

print("master (overscan): bias", float(np.median(master_bias_ov.data)),
      "| dark", float(np.median(master_dark_ov.data)),
      "| flat", float(np.median(master_flat_ov.data)))

ccd_ov_sci = ccdproc.subtract_bias(sci_ovs[0], master_bias_ov)
ccd_ov_sci = ccdproc.subtract_dark(ccd_ov_sci, master_dark_ov,
                                   exposure_time='EXPTIME', exposure_unit=u.second,
                                   scale=True)
ccd_ov_sci = ccdproc.flat_correct(ccd_ov_sci, master_flat_ov)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
d_raw = sci_ovs[0].data
axes[0].imshow(d_raw, cmap='gray', origin='lower',
               vmin=np.percentile(d_raw, 5), vmax=np.percentile(d_raw, 95))
axes[0].set_title('RAW (dengan strip overscan)')
axes[1].imshow(ccd_ov_sci.data, cmap='gray', origin='lower',
               vmin=np.percentile(ccd_ov_sci.data, 5),
               vmax=np.percentile(ccd_ov_sci.data, 95))
axes[1].set_title('TERKALIBRASI (overscan + bias + dark + flat)')
plt.show()

print("median latar belakang terkalibrasi:", float(np.median(ccd_ov_sci.data)))

### ✋ Hands-on 6 (± 8 menit)

1. Plot level bias (median strip) untuk semua 15 frame bias/dark/flat —
   lihat sendiri geseran level antar frame.
2. Kalibrasi frame science kedua (`field-mock-ov-v-120s-002.fit`) dengan
   mini-pipeline yang sama.
3. (Bonus) Ulangi kalibrasi **tanpa** koreksi overscan (master bias langsung
   dari frame mentah, strip ikut terbawa) — bandingkan median latar
   belakang keduanya. Mana yang lebih konsisten, dan kenapa?

In [ ]:
# TODO 1: level bias (median strip) semua frame overscan
# for i in range(1, 6): ...

# TODO 2: kalibrasi frame science kedua
# ccd_sci2 = ...

# TODO 3: kalibrasi tanpa koreksi overscan -> bandingkan latar belakang
# (hati-hati: frame mentah berbentuk (512,528), frame tertrim (512,512))

## Penutup

Hari ini anda, tanpa latar belakang astronomi, sudah:

✅ membaca citra ilmiah (FITS) dan citra biasa (PNG) sebagai array numpy  
✅ menginspeksi metadata dan memahami unit  
✅ mengarakterisasi citra secara statistik; menangani NaN & piksel saturasi  
✅ memahami dan menjalankan kalibrasi bias/dark/flat dengan `ccdproc`  
✅ reduksi derau sederhana & penanganan kanal warna  
✅ mengukur, menyimpan, dan mengekspor hasil $→$ alur kerja reproduktibel  
✅ menangani citra dengan overscan  

### Eksplorasi mandiri

1. Kalibrasi kelima frame science, lalu **gabungkan** (median) hasilnya. 
   Seberapa banyak derau turun?
2. Terapkan alur yang sama ke satu foto PNG anda sendiri. Apa yang menjadi
   "flat" untuk foto ponsel?
3. Dari `meta.groupby(...)`: filter mana yang paling banyak diekspos?

### Referensi

- [Astropy tutorials --- CCD data reduction guide](https://www.astropy.org/ccd-reduction-and-photometry-guide/v/dev/notebooks/01-00-Understanding-an-astronomical-CCD-image.html)
- [`ccdproc` documentation](https://ccdproc.readthedocs.io/en/v1.3.0/)
- [Pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)